# Accuracy test

In [1]:
import pandas as pd
import json
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from functions import *
from LLM_functions import *

#Package for accuracy 
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
fig_path = '/home/lhasbini/como_school/figure/'
file_path = '/scratchx/lhasbini/como_school/filtered_report_types_nat_hazards_summary-header.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    filtered_reports = json.load(json_file)

#unique_hazards = ['Drought', 'Flood', 'Storm', 'Tornado', 'Storm surge', 'Heatwave', 'Coldwave', 'Mass movement', 'Cyclone', 'Tidal Wave', 'Wildfire'] 
#unique_countries_ISO = [country.alpha_3 for country in pycountry.countries]

In [3]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Kenya",
        "Locations": "Nairobi",
        "Start_Date": "March-April-May",
        "End_Date": "NULL",
        "Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    },
    {
        "Hazard": "Landslide",
        "Name": "NULL",
        "Country": "Kenya",
        "Locations": "NULL",
        "Start_Date": "NULL",
        "End_Date": "NULL",
        "Hazard_Sentences": "There were also cases of landslides and mudslides in central Kenya affecting both families with even young children."
    },
    {
        "Hazard": "Drought",
        "Name": "NULL",
        "Country": "Kenya",
        "Locations": "NULL",
        "Start_Date": "four decades",
        "End_Date": "NULL",
        "Hazard_Sentences": "The floods of 2023 floods and these 2024 floods are exacerbating the humanitarian crisis as part of the country have just emerged from the worst drought in four decades, which has left millions of people hungry."
    } 
]

In [4]:
dict_test_labelled = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Kenya",
        "Locations": "Nairobi",
        "Start_Date": "March-April-May",
        "End_Date": "NULL",
        "Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    },
    {
        "Hazard": "Cyclone",
        "Name": "NULL",
        "Country": "Kenya",
        "Locations": "Central Kenya",
        "Start_Date": "NULL",
        "End_Date": "NULL",
        "Hazard_Sentences": "There were also cases of landslides and mudslides in central Kenya affecting both families with even young children."
    },
    {
        "Hazard": "Drought",
        "Name": "NULL",
        "Country": "Cameroon",
        "Locations": "NULL",
        "Start_Date": "four decades",
        "End_Date": "NULL",
        "Hazard_Sentences": "The floods of 2023 floods and these 2024 floods are exacerbating the humanitarian crisis as part of the country have just emerged from the worst drought in four decades, which has left millions of people hungry."
    } 
]

In [5]:
def extract_outer_json(text):
    start_index = text.find('{')
    end_index = text.rfind('}')

    if start_index == -1 or end_index == -1 or start_index >= end_index:
        return None  # Return None for empty JSON or invalid format

    extracted_json = text[start_index:end_index + 1]
    return extracted_json

In [6]:
df_output = pd.DataFrame(dict_output)
df_output['appealCode'] = 'MDRDZ011'

df_test_labelled = pd.DataFrame(dict_test_labelled)
df_test_labelled['appealCode'] = 'MDRDZ011'

In [7]:
#Convert countries to ISO code 3 
df_output['Country'] = [country_name_to_iso3(cntr) for cntr in df_output['Country']]
df_test_labelled['Country'] = [country_name_to_iso3(cntr) for cntr in df_test_labelled['Country']]

In [8]:
unique_dict = {
    'Hazard' : unique_hazards, 
    'Country' : unique_countries_ISO
}

In [9]:
columns_accuracy = ['Hazard', 'Country', 'Locations', 'Start_Date', 'End_Date']#['Hazard_Type', 'Hazard_Count', 'Country']
precision = calculate_precision(df_output, df_test_labelled, columns_accuracy)

In [10]:
precision

,Hazard,Country,Locations,Start_Date,End_Date
0,0.816497,0.707107,0.5,1.0,NaN


In [11]:
df_output

,Hazard,Name,Country,Locations,Start_Date,End_Date,Hazard_Sentences,appealCode
0,Flood,NULL,KEN,Nairobi,March-April-May,NULL,Communities in Kenya are once again facing hea...,MDRDZ011
1,Landslide,NULL,KEN,NULL,NULL,NULL,There were also cases of landslides and mudsli...,MDRDZ011
2,Drought,NULL,KEN,NULL,four decades,NULL,The floods of 2023 floods and these 2024 flood...,MDRDZ011


In [12]:
df_test_labelled

,Hazard,Name,Country,Locations,Start_Date,End_Date,Hazard_Sentences,appealCode
0,Flood,NULL,KEN,Nairobi,March-April-May,NULL,Communities in Kenya are once again facing hea...,MDRDZ011
1,Cyclone,NULL,KEN,Central Kenya,NULL,NULL,There were also cases of landslides and mudsli...,MDRDZ011
2,Drought,NULL,CMR,NULL,four decades,NULL,The floods of 2023 floods and these 2024 flood...,MDRDZ011


# Prepare test examples 

In [17]:
filtered_reports[0]['appealCode']

'MDRDZ011'

In [ ]:
output_json=[
    {
      "Hazard": "Flood",
      "Country" : "Algeria", 
      "Locations": ["southern and western Algeria", "Bchar, Elbayadh, Beni Abbes, Tamanrasset, Tiaret, Tindouf, and Naama"],
      "Start_Date": "September 5, 2024",
      "End_Date": "September 8, 2024",
      "Name": "" 
    }
]

In [39]:
filtered_reports[1]['appealCode']
#filtered_reports[1]['header']

'MDRPK026'

In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Pakistan ",
        "Locations": ["Balochistan ", "Sindh ", "Punjab", "Khyber Pakhtunkhwa KP", "Azad Jammu", "Khyber Pakhtunkhwa KP", 
                     "Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sanghar, Dadu, Shaheed Benazirabad, and Kashmor", 
                     "Taluka Tando Adam"],
        "Start_Date": "July 2024",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    },
    {
        "Hazard": "Mass movement",
        "Name": "NULL",
        "Country": "Pakistan",
        "Locations": ["KP, Azad Jammu and Kashmir AJK, and GilgitBaltistan GB"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
        #"Hazard_Sentences": "There were also cases of landslides and mudslides in central Kenya affecting both families with even young children."
    },
    {
        "Hazard": "Heat Wave",
        "Name": "NULL",
        "Country": "Pakistan",
        "Locations": "NULL",
        "Start_Date": "26 August 2024",
        "End_Date": "1 September 2024",
        #"Hazard_Sentences": "There were also cases of landslides and mudslides in central Kenya affecting both families with even young children."
    }
]

In [40]:
filtered_reports[2]['appealCode']
#filtered_reports[2]['header']

'MDRCM039'

In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Cameroon ",
        "Locations": ["Cameroons Far North region", "Logone et Chari and Mayo Danay", "Yagoua", "Blangoua, Mackary, and Zina", 
                      "Chari division", "Maga, Yagoua", "Logone division", "Ndoukoula district"],
        "Start_Date": "second half of July 2024",
        "End_Date": "August 28, 2024",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    },
    {
        "Hazard": "Drought",
        "Name": "NULL",
        "Country": "Cameroon",
        "Locations": "NULL",
        "Start_Date": "2024",
        "End_Date": "NULL",
        #"Hazard_Sentences": "There were also cases of landslides and mudslides in central Kenya affecting both families with even young children."
    }
]

In [42]:
filtered_reports[3]['appealCode']
#filtered_reports[3]['header']

'MDRBJ019'

In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Benin ",
        "Locations": ["Mono, Couffo, Zou and Oum in the South of Benin", "Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli", 
                      "Ahouada, Hazin, Yamontou, Ahomadegbe, Gnizounme, Hangbannou, Tandji, Aboti, Zounhome, Hehokpa, Sawanou, Tohou Centre and Adjassagon", "
                     ],
        "Start_Date": "26 June 2024",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }
]

In [41]:
print(filtered_reports[4]['appealCode'])
#print(filtered_reports[4]['header'])

MDRSD034


In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Sudan",
        "Locations": ["Red Sea, River Nile, and Northern State" 
                     ],
        "Start_Date": "1 June 2024",
        "End_Date": "12 August 2024",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }
]

In [43]:
print(filtered_reports[5]['appealCode'])
#print(filtered_reports[5]['header'])

MDRNG041


In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Nigeria",
        "Locations": ["Kano", "Maiduguri", "Bauchi state", "Bauchi, Kebbi, Sokoto, Zamfara"
                     ],
        "Start_Date": "8 August 2024",
        "End_Date": "13 August 2024",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Nigeria",
        "Locations": ["Sokoto State", "Dantudu, Balakozo, Gidan Tudu, and Tsitse", "Zamfara State", "Ruwan Gora, Morai, Makera, and Talata Mafara town"],
        "Start_Date": "17 July 2024",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }
]

In [44]:
print(filtered_reports[6]['appealCode'])
#print(filtered_reports[6]['header'])

MDRRW022


In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Rwanda",
        "Locations": ["western, northern and southern provinces", "14 districts"
                     ],
        "Start_Date": "1 May 2023",
        "End_Date": "June 2023",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
    {
        "Hazard": "Mass Movement",
        "Name": "NULL",
        "Country": "Rwanda",
        "Locations": ["14 districts"
                     ],
        "Start_Date": "1 May 2023",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
]

In [45]:
print(filtered_reports[7]['appealCode'])
#print(filtered_reports[7]['header'])

MDRZM022


In [ ]:
dict_output = [
    {
        "Hazard": "Drought",
        "Name": "NULL",
        "Country": "Zambia",
        "Locations": ["Lusaka, Luapula, and the Western, Southern, Central, and Northwestern Provinces", "Western, Southern, and NorthWestern."
                     ],
        "Start_Date": "29 February 2024",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }
]

In [38]:
print(filtered_reports[8]['appealCode'])
#print(filtered_reports[8]['header'])

MDRUG050


In [ ]:
dict_output = [
    {
        "Hazard": "Flood",
        "Name": "NULL",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa", 
                      "Manafwa, Lwakhakha, Sironko, Mpologoma, Awoja, Nbuyonga, and Namatala"],
        "Start_Date": "April 2024",
        "End_Date": "31st August 2024",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
    {
        "Hazard": "Storm",
        "Name": "NULL",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
    {
        "Hazard": "Mass Movement",
        "Name": "NULL",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }
]

In [37]:
print(filtered_reports[9]['appealCode'])
#print(filtered_reports[9]['header'])

MDRMZ024


In [ ]:
dict_output = [
    {
        "Hazard": "Cyclone",
        "Name": "Filipo",
        "Country": "Mozambique",
        "Locations": [],
        "Start_Date": "March 2024",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
    {
        "Hazard": "Cyclone",
        "Name": "Freddy",
        "Country": "Mozambique",
        "Locations": [],
        "Start_Date": "2023",
        "End_Date": "NULL",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }, 
    {
        "Hazard": "Drought",
        "Name": "Freddy",
        "Country": "Mozambique",
        "Locations": ["central and northern zones"],
        "Start_Date": "May 2024",
        "End_Date": "June 2024",
        #"Hazard_Sentences": "Communities in Kenya are once again facing heavy rains and devastating floods. The floods have also hit the capital in residential informal settlements areas of Nairobi as rivers overflowed."
    }
]

In [ ]:
hazard_patterns_hw = {
'Drought': r"drought.|dry spell.", 
'Flood': r"\b(flood|floods|flooding|inundation|inundations|glacial lake outburst)\b",
'Storm': r"storm.|superstorm.|tornado.|windstorm.|snowstorm.|snowfal.|blizzard.|derecho.|winterstorm.|hail.|extra tropical storm.|thunderstorm.",
'Tornado': r"tornado.*",
'Storm surge': r"storm surge.*",
'Heatwave': r"heat wave.|heatwave.|heat episode.|((heat|hot) spell).|heat stress.*|high temperature.*",
'Coldwave': r"cold wave.|coldwave.|severe winter conditions.|cold spell.",
'Mass movement': r"land slide.|landslide.|rockfall.|mudslide.|mass movement.*",
'Cyclone': r"cyclone.|tropical cyclone.|hurricane.|typhoon.",
'Tidal Wave': r"tidal wav.*",
'Wildfire': r"fire.|forestfire.|wildfire.|landfire.|bushfire.|forest fire.|wild fire.|land fire.|bush fire.*" }